In [ ]:
!pip install catboost
!pip install lime

In [ ]:
# =============================================================================
# HYPERPARAMETER TUNING (RandomizedSearchCV)
# Delivery Delay Severity Prediction (Regression)
# =============================================================================

from sklearn.model_selection import RandomizedSearchCV
import time

RANDOM_STATE = 42
N_JOBS = -1 # Define N_JOBS for parallel processing


param_grids = {

    # ── Decision Tree ────────────────────────────────────────────────────────
    "Decision Tree": {
        "model__max_depth"        : [4, 6, 8, 10, 12, None],
        "model__min_samples_leaf" : [10, 20, 30, 50],
        "model__min_samples_split": [20, 40, 60],
        "model__criterion"        : ["squared_error", "friedman_mse"],
    },

    # ── Random Forest ────────────────────────────────────────────────────────
    "Random Forest": {
        "model__n_estimators"     : [100, 200, 300],
        "model__max_depth"        : [6, 8, 10, None],
        "model__min_samples_leaf" : [10, 20, 30],
        "model__min_samples_split": [20, 40, 60],
        "model__max_features"     : ["sqrt", "log2", 0.8],
    },

    # ── XGBoost ──────────────────────────────────────────────────────────────
    "XGBoost": {
        "model__n_estimators"     : [200, 300, 500],
        "model__learning_rate"    : [0.01, 0.03, 0.05, 0.1],
        "model__max_depth"        : [4, 6, 8],
        "model__subsample"        : [0.7, 0.8, 1.0],
        "model__colsample_bytree" : [0.7, 0.8, 1.0],
        "model__reg_alpha"        : [0, 0.1, 0.5],
        "model__reg_lambda"       : [1.0, 2.0, 5.0],
    },

    # ── LightGBM ─────────────────────────────────────────────────────────────
    "LightGBM": {
        "model__n_estimators"      : [200, 300, 500],
        "model__learning_rate"     : [0.01, 0.03, 0.05, 0.1],
        "model__num_leaves"        : [31, 63, 127],
        "model__max_depth"         : [4, 6, 8, -1],
        "model__subsample"         : [0.7, 0.8, 1.0],
        "model__reg_alpha"         : [0, 0.1, 0.5],
        "model__reg_lambda"        : [1.0, 2.0, 5.0],
        "model__min_child_samples" : [20, 30, 50],
    },

    # ── CatBoost ─────────────────────────────────────────────────────────────
    "CatBoost": {
        "model__iterations"          : [200, 300, 500],
        "model__learning_rate"       : [0.01, 0.03, 0.05, 0.1],
        "model__depth"               : [4, 6, 8],
        "model__l2_leaf_reg"         : [1, 3, 5, 7],
        "model__bagging_temperature" : [0.0, 0.5, 1.0],
    },

    # ── SVR ──────────────────────────────────────────────────────────────────

    "SVR": {
        "model__C"      : [0.1, 1.0, 5.0, 10.0],
        "model__epsilon": [0.1, 0.5, 1.0, 2.0],
        "model__gamma"  : ["scale", "auto"],
        "model__kernel" : ["rbf"],
    },
}

# =============================================================================
# ITERATION CONTROL
# =============================================================================

n_iter_map = {
    "Decision Tree" : 5,
    "Random Forest" : 5,
    "XGBoost"       : 7,
    "LightGBM"      : 7,
    "CatBoost"      : 7,
    "SVR"           : 5,
}

# =============================================================================
# CV STRATEGY
# =============================================================================

tuning_kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# =============================================================================
# RANDOMIZED SEARCH TUNING LOOP — TREE MODELS
# =============================================================================

print("\n")
print("=" * 100)
print("  HYPERPARAMETER TUNING — TASK 2: DELIVERY DELAY SEVERITY PREDICTION")
print("=" * 100)


tuned_results = {}

# Tree models — use full X_train
# Linear Regression skipped — no meaningful hyperparameters to tune
models_to_tune = {
    k: v for k, v in models.items()
    if k not in ["Linear Regression", "SVR"]
}

for model_name, pipeline in models_to_tune.items():

    print(f"\n   Tuning → {model_name} ...", end=" ", flush=True)
    t0 = time.time()

    rs = RandomizedSearchCV(
        estimator           = pipeline,
        param_distributions = param_grids[model_name],
        n_iter              = n_iter_map[model_name],
        cv                  = tuning_kf,
        # Regression equivalent of f1_macro — lower MAE = better model
        scoring             = "neg_mean_absolute_error",
        n_jobs              = N_JOBS,
        random_state        = RANDOM_STATE,
        verbose             = 0,
        refit               = True,
        return_train_score  = False,
    )

    rs.fit(X_train, y_train)
    train_time    = time.time() - t0
    best_pipeline = rs.best_estimator_

    # Full test set evaluation after tuning
    y_pred = best_pipeline.predict(X_test)
    mae    = mean_absolute_error(y_test, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    r2     = r2_score(y_test, y_pred)

    # CV MAE from tuning search (negated back to positive)
    cv_mae_tuned = abs(rs.best_score_)

    tuned_results[model_name] = {
        # Full metric suite
        "MAE"            : round(mae,          4),
        "RMSE"           : round(rmse,         4),
        "R2"             : round(r2,           4),
        "CV MAE (tuned)" : round(cv_mae_tuned, 4),
        "Best Params"    : rs.best_params_,
        "Train Time (s)" : round(train_time,   1),
        "_pipeline"      : best_pipeline,
        "_y_pred"        : y_pred,
    }

    print(
        f"done [{train_time:.1f}s]  "
        f"MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}"
    )

# =============================================================================
# SVR TUNING — SEPARATE BLOCK
# =============================================================================

print(f"\n   Tuning → SVR (sampled) ...", end=" ", flush=True)

# Use the same SVR sample indices from main code
X_svr_tune = X_train.iloc[svr_sample_idx]
y_svr_tune = y_train.iloc[svr_sample_idx]

t0 = time.time()

rs_svr = RandomizedSearchCV(
    estimator           = svr_pipeline,
    param_distributions = param_grids["SVR"],
    n_iter              = n_iter_map["SVR"],
    cv                  = tuning_kf,
    scoring             = "neg_mean_absolute_error",
    n_jobs              = N_JOBS,
    random_state        = RANDOM_STATE,
    verbose             = 0,
    refit               = True,
    return_train_score  = False,
)

rs_svr.fit(X_svr_tune, y_svr_tune)
train_time    = time.time() - t0
best_svr_pipe = rs_svr.best_estimator_

y_pred_svr = best_svr_pipe.predict(X_test)
mae_svr    = mean_absolute_error(y_test, y_pred_svr)
rmse_svr   = np.sqrt(mean_squared_error(y_test, y_pred_svr))
r2_svr     = r2_score(y_test, y_pred_svr)
cv_mae_svr = abs(rs_svr.best_score_)

tuned_results["SVR"] = {
    "MAE"            : round(mae_svr,    4),
    "RMSE"           : round(rmse_svr,   4),
    "R2"             : round(r2_svr,     4),
    "CV MAE (tuned)" : round(cv_mae_svr, 4),
    "Best Params"    : rs_svr.best_params_,
    "Train Time (s)" : round(train_time, 1),
    "_pipeline"      : best_svr_pipe,
    "_y_pred"        : y_pred_svr,
}

print(
    f"done [{train_time:.1f}s]  "
    f"MAE={mae_svr:.4f}  RMSE={rmse_svr:.4f}  R²={r2_svr:.4f}"
)

# Linear Regression — no tuning, carry forward from original results
lr_entry = results["Linear Regression"].copy()
lr_entry["CV MAE (tuned)"] = lr_entry["CV MAE"]   # alias for consistent key
tuned_results["Linear Regression"] = lr_entry
print("\n   Linear Regression — no hyperparameters to tune, carried forward.")

# =============================================================================
# BEFORE vs AFTER TUNING COMPARISON
# =============================================================================

print("\n")
print("=" * 100)
print("  TUNING COMPARISON — PRE vs POST RandomizedSearchCV (TASK 2)")
print("=" * 100)

rows = []
for m in tuned_results:
    rows.append({
        "Model"          : m,
        "MAE (pre)"      : results[m]["MAE"],
        "MAE (tuned)"    : tuned_results[m]["MAE"],
        "RMSE (pre)"     : results[m]["RMSE"],
        "RMSE (tuned)"   : tuned_results[m]["RMSE"],
        "R² (pre)"       : results[m]["R2"],
        "R² (tuned)"     : tuned_results[m]["R2"],
        "Train Time (s)" : tuned_results[m]["Train Time (s)"],
    })

compare_df_tuning = pd.DataFrame(rows).sort_values(
    "MAE (tuned)", ascending=True       # lower MAE = better for regression
).reset_index(drop=True)

print(compare_df_tuning.to_string(index=False))
print("=" * 100)

compare_df_tuning.to_csv(
    "outputs_task2/task2_tuning_comparison.csv", index=False
)
print("\n[✓] Saved → outputs_task2/task2_tuning_comparison.csv")

# =============================================================================
# SAVE BEST HYPERPARAMETERS
# =============================================================================

params_rows = [
    {"Model": m, "Best Params": str(tuned_results[m].get("Best Params", "N/A"))}
    for m in tuned_results
]
pd.DataFrame(params_rows).to_csv(
    "outputs_task2/task2_best_params.csv", index=False
)
print("[✓] Saved → outputs_task2/task2_best_params.csv")

# Print best params — all models
print("\n── Best Hyperparameters — All Models ───────────────────────────────────")
for m in tuned_results:
    if tuned_results[m].get("Best Params"):
        print(f"\n   {m}:")
        for param, val in tuned_results[m]["Best Params"].items():
            print(f"     {param:<40} : {val}")
    else:
        print(f"\n   {m}: No tuning performed (Linear Regression)")

# =============================================================================
# FINAL TUNED MODEL RANKING
# =============================================================================

print("\n")
print("=" * 100)
print("  FINAL TUNED MODEL RANKING — TASK 2: DELIVERY DELAY SEVERITY PREDICTION")
print("=" * 100)

final_tuned_df = pd.DataFrame([
    {
        "Model"          : m,
        "MAE"            : tuned_results[m]["MAE"],
        "RMSE"           : tuned_results[m]["RMSE"],
        "R2"             : tuned_results[m]["R2"],
        "CV MAE (tuned)" : tuned_results[m]["CV MAE (tuned)"],
        "Train Time (s)" : tuned_results[m]["Train Time (s)"],
    }
    for m in tuned_results
])

# Sort ascending by MAE — lower is better for regression
final_tuned_df = final_tuned_df.sort_values(
    "MAE", ascending=True
).reset_index(drop=True)
final_tuned_df.insert(0, "Rank", final_tuned_df.index + 1)

print(final_tuned_df.to_string(index=False))
print("=" * 100)

best_tuned_name     = final_tuned_df.iloc[0]["Model"]
best_tuned_pipeline = tuned_results[best_tuned_name]["_pipeline"]
best_tuned_y_pred   = tuned_results[best_tuned_name]["_y_pred"]

print(f"\n   Best Tuned Model  : {best_tuned_name}")
print(f"   MAE               : {tuned_results[best_tuned_name]['MAE']:.4f}")
print(f"   RMSE              : {tuned_results[best_tuned_name]['RMSE']:.4f}")
print(f"   R²                : {tuned_results[best_tuned_name]['R2']:.4f}")
print(f"   Best Params       : {tuned_results[best_tuned_name].get('Best Params', 'N/A')}")
print("=" * 100)

final_tuned_df.to_csv(
    "outputs_task2/task2_tuned_model_comparison.csv", index=False
)
print("\n[✓] Saved → outputs_task2/task2_tuned_model_comparison.csv")

# =============================================================================
# POST-TUNING VISUALISATIONS
# =============================================================================

print("\n── Post-Tuning Visualisations ───────────────────────────────────────────")

PALETTE_TUNED = sns.color_palette("Set2", n_colors=len(tuned_results))
MODEL_ORDER_TUNED = final_tuned_df["Model"].tolist()

# ── Grouped bar chart — MAE, RMSE, R² (tuned) ────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
bar_metrics = ["MAE", "RMSE", "R2"]
bar_colors  = ["#EF5350", "#FFA726", "#43A047"]
x     = np.arange(len(MODEL_ORDER_TUNED))
width = 0.25

for i, (metric, color) in enumerate(zip(bar_metrics, bar_colors)):
    vals = [tuned_results[m][metric] for m in MODEL_ORDER_TUNED]
    bars = ax.bar(x + i * width, vals, width,
                  label=metric, color=color, alpha=0.85)
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f"{bar.get_height():.3f}",
            ha="center", va="bottom", fontsize=7, rotation=90
        )

ax.set_xticks(x + width)
ax.set_xticklabels(MODEL_ORDER_TUNED, rotation=25, ha="right", fontsize=9)
ax.set_ylabel("Score")
ax.set_title(
    "Task 2 — Tuned Model Comparison: MAE | RMSE | R²",
    fontsize=13, fontweight="bold"
)
ax.legend(loc="upper right", fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(
    "outputs_task2/task2_tuned_model_comparison_bars.png",
    dpi=150, bbox_inches="tight"
)
plt.close()
print("   [✓] Saved outputs_task2/task2_tuned_model_comparison_bars.png")

# ── Heatmap — tuned model metrics ─────────────────────────────────────────────
hm_cols = ["MAE", "RMSE", "R2", "CV MAE (tuned)"]
hm_df   = final_tuned_df.set_index("Model")[hm_cols].astype(float)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    hm_df, annot=True, fmt=".4f",
    cmap="YlOrRd_r",          # reversed — lower MAE = darker = better
    linewidths=0.5, ax=ax,
    annot_kws={"size": 9}
)
ax.set_title(
    "Task 2 — Tuned Model Performance Heatmap",
    fontsize=13, fontweight="bold"
)
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(
    "outputs_task2/task2_tuned_heatmap.png",
    dpi=150, bbox_inches="tight"
)
plt.close()
print("   [✓] Saved outputs_task2/task2_tuned_heatmap.png")

# ── Actual vs Predicted scatter — best tuned model ────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(
    y_test, best_tuned_y_pred,
    alpha=0.3, s=8, color="#1976D2"
)
mn = min(float(y_test.min()), float(best_tuned_y_pred.min()))
mx = max(float(y_test.max()), float(best_tuned_y_pred.max()))
ax.plot([mn, mx], [mn, mx], "r--", lw=1.5, label="Perfect prediction")
ax.set_xlabel("Actual Delay Days")
ax.set_ylabel("Predicted Delay Days")
ax.set_title(
    f"Task 2 — Actual vs Predicted: {best_tuned_name} (Tuned)\n"
    f"MAE={tuned_results[best_tuned_name]['MAE']:.4f}  "
    f"RMSE={tuned_results[best_tuned_name]['RMSE']:.4f}  "
    f"R²={tuned_results[best_tuned_name]['R2']:.4f}",
    fontsize=10, fontweight="bold"
)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    "outputs_task2/task2_tuned_actual_vs_predicted.png",
    dpi=150, bbox_inches="tight"
)
plt.close()
print("   [✓] Saved outputs_task2/task2_tuned_actual_vs_predicted.png")

# ── Residual plot — best tuned model ──────────────────────────────────────────
residuals = np.array(y_test) - best_tuned_y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Residuals vs predicted
axes[0].scatter(best_tuned_y_pred, residuals,
                alpha=0.3, s=8, color="#7B1FA2")
axes[0].axhline(0, color="red", linestyle="--", lw=1.5)
axes[0].set_xlabel("Predicted Delay Days")
axes[0].set_ylabel("Residuals (Actual - Predicted)")
axes[0].set_title(
    f"Residuals vs Predicted — {best_tuned_name} (Tuned)",
    fontsize=10, fontweight="bold"
)
axes[0].grid(alpha=0.3)

# Residual distribution
axes[1].hist(residuals, bins=30, color="#7B1FA2",
             edgecolor="white", alpha=0.85)
axes[1].axvline(0, color="red", linestyle="--", lw=1.5)
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Frequency")
axes[1].set_title(
    "Residual Distribution",
    fontsize=10, fontweight="bold"
)
axes[1].grid(alpha=0.3)

fig.suptitle(
    f"Task 2 — Residual Analysis: {best_tuned_name} (Tuned)",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(
    "outputs_task2/task2_tuned_residuals.png",
    dpi=150, bbox_inches="tight"
)
plt.close()
print("   [✓] Saved outputs_task2/task2_tuned_residuals.png")

# ── CV MAE horizontal bar chart — tuned models ────────────────────────────────
print("\n── Post-Tuning Cross-Validation MAE (5-Fold) ───────────────────────────")

tuned_cv_means = []
tuned_cv_stds  = []

for model_name in MODEL_ORDER_TUNED:
    pipe = tuned_results[model_name]["_pipeline"]

    # SVR uses sampled data for CV — consistent with main code
    if model_name == "SVR":
        X_cv, y_cv = X_svr_tune, y_svr_tune
    else:
        X_cv, y_cv = X_train, y_train

    cv_s = cross_val_score(
        pipe, X_cv, y_cv,
        cv=tuning_kf,
        scoring="neg_mean_absolute_error",
        n_jobs=N_JOBS
    )
    mean_mae = abs(cv_s.mean())
    std_mae  = cv_s.std()
    tuned_cv_means.append(mean_mae)
    tuned_cv_stds.append(std_mae)
    print(f"   {model_name:<22}  CV MAE = {mean_mae:.4f} ± {std_mae:.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(
    MODEL_ORDER_TUNED, tuned_cv_means,
    xerr=tuned_cv_stds,
    color=PALETTE_TUNED[:len(MODEL_ORDER_TUNED)],
    height=0.55, capsize=5, ecolor="gray"
)
ax.set_xlabel("CV MAE (mean ± std)  —  lower is better")
ax.set_title(
    "Task 2 — Cross-Validated MAE: Tuned Models (5-Fold)",
    fontsize=12, fontweight="bold"
)
for i, (m, s) in enumerate(zip(tuned_cv_means, tuned_cv_stds)):
    ax.text(m + s + 0.005, i, f"{m:.4f}", va="center", fontsize=8)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(
    "outputs_task2/task2_tuned_cv_mae.png",
    dpi=150, bbox_inches="tight"
)
plt.close()
print("[✓] Saved outputs_task2/task2_tuned_cv_mae.png")

# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n")
print("=" * 100)
print("  FINAL SUMMARY — TASK 2: POST-TUNING RESULTS")
print("=" * 100)
print(final_tuned_df[[
    "Rank", "Model", "MAE", "RMSE", "R2",
    "CV MAE (tuned)", "Train Time (s)"
]].to_string(index=False))
print("=" * 100)
print(f"\n   Best Tuned Model  : {best_tuned_name}")
print(f"   MAE               : {tuned_results[best_tuned_name]['MAE']:.4f}")
print(f"   RMSE              : {tuned_results[best_tuned_name]['RMSE']:.4f}")
print(f"   R²                : {tuned_results[best_tuned_name]['R2']:.4f}")
print(f"\n   Saved outputs:")
print("    • outputs_task2/task2_tuning_comparison.csv")
print("    • outputs_task2/task2_best_params.csv")
print("    • outputs_task2/task2_tuned_model_comparison.csv")
print("    • outputs_task2/task2_tuned_model_comparison_bars.png")
print("    • outputs_task2/task2_tuned_heatmap.png")
print("    • outputs_task2/task2_tuned_actual_vs_predicted.png")
print("    • outputs_task2/task2_tuned_residuals.png")
print("    • outputs_task2/task2_tuned_cv_mae.png")
print("=" * 100)
